In [ ]:
# ===============================

In [ ]:
# 10 Runs

In [1]:
"""
==========================================================================
ADVANCED BRAIN TUMOR MRI TRAINING - PATENT DOCUMENTATION (10 RUNS EDITION)
==========================================================================
يطابق منهجية الـ Holdout الموصوفة:
- تُجمع كل الصور (من مجلدي Training و Testing معاً إن وُجدا، أو من مجلد
  واحد فقط) في مجموعة بيانات واحدة (6420 صورة في حالتك).
- تُقسَّم عشوائياً مرة واحدة: 80% Training / 20% Testing.
- لا يوجد Validation منفصل: النموذج يتدرب على الـ80% ويُقيَّم على الـ20%
  المتبقية فقط (Holdout الكلاسيكي كما هو موصوف في النص).
- يُنفَّذ التدريب 10 مرات (10 Runs) بتقسيمات عشوائية مختلفة في كل مرة
  (seed مختلف) لإثبات استقرار الأداء، ثم يُحسب المتوسط والانحراف المعياري
  للمقاييس عبر كل الـ Runs.
==========================================================================
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve, auc,
                              accuracy_score, precision_score, recall_score, f1_score)
from sklearn.preprocessing import label_binarize
import time
from tqdm import tqdm
import warnings
import gc
import os
import json
import random
from pathlib import Path
warnings.filterwarnings('ignore')

plt.ioff()
import matplotlib
matplotlib.use('Agg')
plt.style.use('default')
sns.set_palette("husl")

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

try:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if torch.cuda.is_available():
        torch.cuda.set_per_process_memory_fraction(0.8)
        print(f"GPU: {torch.cuda.get_device_name()}")
except Exception:
    device = torch.device('cpu')
print(f"Using device: {device}")


# ==========================================================================
# Reproducibility helper
# ==========================================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ==========================================================================
# Dataset Class - يجمع كل الصور من Training و/أو Testing في مجموعة واحدة
# ==========================================================================
class BrainTumorMRIDataset(Dataset):
    """
    يقرأ كل الصور المتاحة تحت data_dir (سواء كانت داخل مجلد Training فقط،
    أو Training + Testing معاً) ويجمعها في مجموعة بيانات واحدة موحّدة،
    تمهيداً لتطبيق Holdout 80/20 عليها بأكملها كما هو موصوف في المنهجية.
    """
    def __init__(self, data_dir, transform=None):
        self.data_dir = Path(data_dir)
        self.transform = transform

        self.class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']
        self.class_to_idx = {c: i for i, c in enumerate(self.class_names)}

        self.samples = []
        self._scan_directory()

        print(f"📁 إجمالي الصور المجمّعة (كل المصادر): {len(self.samples)}")
        self._print_class_distribution()

    def _scan_directory(self):
        # يبحث عن أي مجلد فرعي يحتوي مجلدات الفئات مباشرة (مثل Training, Testing)
        # أو عن الفئات مباشرة تحت data_dir نفسه
        candidate_roots = []

        for sub in ["Training", "Testing"]:
            p = self.data_dir / sub
            if p.exists():
                candidate_roots.append(p)

        # إذا لم يوجد Training/Testing، جرّب data_dir نفسه مباشرة
        if not candidate_roots:
            candidate_roots.append(self.data_dir)

        seen_paths = set()
        for root in candidate_roots:
            print(f"📂 Scanning: {root}")
            for class_name in self.class_names:
                class_folder = root / class_name
                if class_folder.exists():
                    label = self.class_to_idx[class_name]
                    for img_path in class_folder.glob('*'):
                        if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                            p = str(img_path.resolve())
                            if p not in seen_paths:
                                seen_paths.add(p)
                                self.samples.append((p, label))

        print(f"✅ تم تحميل {len(self.samples)} صورة فريدة من كل المصادر المتاحة")

    def _print_class_distribution(self):
        counts = [0] * len(self.class_names)
        for _, label in self.samples:
            counts[label] += 1
        print("📊 Class Distribution (Full Dataset):")
        for name, c in zip(self.class_names, counts):
            print(f"  {name}: {c} images")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            black_img = Image.new('RGB', (128, 128), (0, 0, 0))
            if self.transform:
                return self.transform(black_img), label
            return torch.zeros(3, 128, 128), label


# ==========================================================================
# Patent Components (unchanged)
# ==========================================================================
class MicroAdaptiveFeatureExtractor(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        reduced_ch = max(1, channels // reduction)
        self.spatial_conv = nn.Conv2d(channels, 1, 1)
        self.channel_avg = nn.AdaptiveAvgPool2d(1)
        self.channel_fc = nn.Conv2d(channels, reduced_ch, 1)
        self.channel_out = nn.Conv2d(reduced_ch, channels, 1)
        self.alpha = nn.Parameter(torch.tensor(0.3), requires_grad=True)

    def forward(self, x):
        spatial_weights = torch.sigmoid(self.spatial_conv(x))
        x_spatial = x * spatial_weights
        channel_pool = self.channel_avg(x_spatial)
        channel_weights = torch.sigmoid(self.channel_out(F.relu(self.channel_fc(channel_pool))))
        x_enhanced = x_spatial * channel_weights
        return x + self.alpha * x_enhanced


class MicroMultiScaleProcessor(nn.Module):
    def __init__(self, channels):
        super().__init__()
        inter = max(8, channels // 4)
        while inter % 4 != 0 and inter > 4:
            inter -= 1
        if inter < 4:
            inter = 4

        self.scale1 = nn.Conv2d(channels, inter, 1, bias=False)
        self.scale3 = nn.Conv2d(channels, inter, 3, padding=1, bias=False)

        if channels >= 8 and inter >= 4:
            groups = 1
            for g in [8, 4, 2]:
                if channels % g == 0:
                    groups = g
                    break
            self.scale3dw = nn.Conv2d(channels, inter, 3, padding=1, groups=groups, bias=False)
        else:
            self.scale3dw = nn.Conv2d(channels, inter, 3, padding=1, bias=False)

        self.fusion = nn.Conv2d(inter * 3, channels, 1, bias=False)
        self.beta = nn.Parameter(torch.tensor(0.3), requires_grad=True)

    def forward(self, x):
        s1 = F.relu(self.scale1(x))
        s3 = F.relu(self.scale3(x))
        s3dw = F.relu(self.scale3dw(x))
        combined = torch.cat([s1, s3, s3dw], dim=1)
        fused = self.fusion(combined)
        return F.relu(x + self.beta * fused)


class MicroIntelligentPooling(nn.Module):
    def __init__(self, channels):
        super().__init__()
        decision_ch = max(2, channels // 8)
        self.decision = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, decision_ch, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(decision_ch, 2, 1),
            nn.Softmax(dim=1)
        )

    def forward(self, x):
        B, C, H, W = x.shape
        decision_weights = self.decision(x)
        max_pool = F.max_pool2d(x, kernel_size=2, stride=2)
        avg_pool = F.avg_pool2d(x, kernel_size=2, stride=2)
        w1 = decision_weights[:, 0:1, :, :].view(B, 1, 1, 1)
        w2 = decision_weights[:, 1:2, :, :].view(B, 1, 1, 1)
        return w1 * max_pool + w2 * avg_pool


# ==========================================================================
# Main Model (unchanged)
# ==========================================================================
class UltraLightPatentBrainTumorCNN(nn.Module):
    def __init__(self, num_classes=4, base_channels=[32, 64, 128, 256]):
        super().__init__()
        c1, c2, c3, c4 = base_channels

        self.conv1 = nn.Conv2d(3, c1, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(c1)
        self.adaptive1 = MicroAdaptiveFeatureExtractor(c1, reduction=4)
        self.pool1 = MicroIntelligentPooling(c1)

        self.conv2 = nn.Conv2d(c1, c2, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(c2)
        self.multiscale2 = MicroMultiScaleProcessor(c2)
        self.pool2 = MicroIntelligentPooling(c2)

        self.conv3 = nn.Conv2d(c2, c3, 3, padding=1, bias=False)
        self.bn3 = nn.BatchNorm2d(c3)
        self.adaptive3 = MicroAdaptiveFeatureExtractor(c3, reduction=8)
        self.pool3 = MicroIntelligentPooling(c3)

        self.conv4 = nn.Conv2d(c3, c4, 3, padding=1, bias=False)
        self.bn4 = nn.BatchNorm2d(c4)
        self.multiscale4 = MicroMultiScaleProcessor(c4)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(c4, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.adaptive1(x)
        x = self.pool1(x)

        x = F.relu(self.bn2(self.conv2(x)))
        x = self.multiscale2(x)
        x = self.pool2(x)

        x = F.relu(self.bn3(self.conv3(x)))
        x = self.adaptive3(x)
        x = self.pool3(x)

        x = F.relu(self.bn4(self.conv4(x)))
        x = self.multiscale4(x)

        return self.classifier(x)


# ==========================================================================
# Training / Evaluation loops
# ==========================================================================
def train_epoch(model, dataloader, optimizer, criterion, device, epoch, num_epochs, run_id):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(dataloader, desc=f"Run {run_id} | Epoch {epoch+1}/{num_epochs} - Train", ncols=100)

    for data, targets in pbar:
        data, targets = data.to(device, non_blocking=True), targets.to(device, non_blocking=True)
        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += targets.size(0)
        correct += (predicted == targets).sum().item()
        pbar.set_postfix({'Loss': f'{loss.item():.3f}', 'Acc': f'{100.*correct/total:.1f}%'})

        del outputs, loss
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    return running_loss / len(dataloader), 100. * correct / total


def evaluate(model, dataloader, criterion, device, desc="Eval"):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_targets, all_probs = [], [], []

    pbar = tqdm(dataloader, desc=desc, ncols=100)
    with torch.no_grad():
        for data, targets in pbar:
            data, targets = data.to(device, non_blocking=True), targets.to(device, non_blocking=True)
            outputs = model(data)
            loss = criterion(outputs, targets)

            probs = F.softmax(outputs, dim=1)
            all_probs.extend(probs.cpu().numpy())

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
            pbar.set_postfix({'Loss': f'{loss.item():.3f}', 'Acc': f'{100.*correct/total:.1f}%'})

            del outputs, loss
            if device.type == 'cuda':
                torch.cuda.empty_cache()

    return (running_loss / len(dataloader), 100. * correct / total,
            all_preds, all_targets, np.array(all_probs))


# ==========================================================================
# Transforms
# ==========================================================================
def create_transforms(image_size=128, is_training=True):
    if is_training:
        return transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.RandomHorizontalFlip(p=0.4),
            transforms.RandomVerticalFlip(p=0.3),
            transforms.RandomRotation(degrees=20),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    else:
        return transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])


# ==========================================================================
# Holdout 80/20 Data loaders (بدون Validation منفصل)
# ==========================================================================
def create_holdout_loaders(full_dataset, batch_size=16, image_size=128,
                            train_split=0.8, seed=42):
    """
    يطبّق Holdout الكلاسيكي: تقسيم عشوائي واحد للمجموعة الكاملة إلى
    80% Training و 20% Testing، دون أي Validation منفصل.
    """
    train_transform = create_transforms(image_size, is_training=True)
    test_transform = create_transforms(image_size, is_training=False)

    total_size = len(full_dataset)
    indices = list(range(total_size))
    np.random.seed(seed)
    np.random.shuffle(indices)

    train_size = int(train_split * total_size)
    train_indices = indices[:train_size]
    test_indices = indices[train_size:]

    train_samples = [full_dataset.samples[i] for i in train_indices]
    test_samples = [full_dataset.samples[i] for i in test_indices]

    train_dataset = BrainTumorMRIDataset.__new__(BrainTumorMRIDataset)
    train_dataset.data_dir = full_dataset.data_dir
    train_dataset.transform = train_transform
    train_dataset.class_names = full_dataset.class_names
    train_dataset.class_to_idx = full_dataset.class_to_idx
    train_dataset.samples = train_samples

    test_dataset = BrainTumorMRIDataset.__new__(BrainTumorMRIDataset)
    test_dataset.data_dir = full_dataset.data_dir
    test_dataset.transform = test_transform
    test_dataset.class_names = full_dataset.class_names
    test_dataset.class_to_idx = full_dataset.class_to_idx
    test_dataset.samples = test_samples

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                               num_workers=0, pin_memory=(device.type == 'cuda'),
                               drop_last=True, persistent_workers=False)

    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                              num_workers=0, pin_memory=(device.type == 'cuda'),
                              persistent_workers=False)

    print(f"📈 Training samples (80%): {len(train_dataset)}")
    print(f"🧪 Testing samples  (20%): {len(test_dataset)}")

    return train_loader, test_loader, full_dataset.class_names


# ==========================================================================
# Plots for ONE run
# ==========================================================================
def plot_run_summary(run_id, train_losses, train_accs, test_acc_per_epoch,
                      test_targets, test_preds, test_probs, class_names, out_dir):
    epochs = range(1, len(train_losses) + 1)

    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_losses, 'b-', marker='o', markersize=3, label='Train Loss')
    plt.title(f'Run {run_id} - Training Loss'); plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.legend(); plt.grid(alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_accs, 'b-', marker='o', markersize=3, label='Train Acc')
    plt.plot(epochs, test_acc_per_epoch, 'g-', marker='^', markersize=3, label='Test Acc (holdout 20%)')
    plt.title(f'Run {run_id} - Accuracy'); plt.xlabel('Epoch'); plt.ylabel('Acc (%)')
    plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{out_dir}/run{run_id}_training_curves.png', dpi=200, facecolor='white')
    plt.close()

    plt.figure(figsize=(8, 7))
    cm = confusion_matrix(test_targets, test_preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Run {run_id} - Confusion Matrix (20% Holdout Test)')
    plt.xlabel('Predicted'); plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(f'{out_dir}/run{run_id}_test_confusion_matrix.png', dpi=200, facecolor='white')
    plt.close()

    if test_probs.size > 0:
        y_test_bin = label_binarize(test_targets, classes=range(len(class_names)))
        plt.figure(figsize=(9, 7))
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
        for i in range(len(class_names)):
            fpr, tpr, _ = roc_curve(y_test_bin[:, i], test_probs[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, color=colors[i % len(colors)], lw=2,
                     label=f'{class_names[i]} (AUC={roc_auc:.3f})')
        plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
        plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
        plt.title(f'Run {run_id} - ROC Curves (20% Holdout Test)')
        plt.legend(loc='lower right'); plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(f'{out_dir}/run{run_id}_test_roc_curves.png', dpi=200, facecolor='white')
        plt.close()


def plot_multi_run_summary(all_run_results, out_dir):
    accs = [r['test_accuracy'] for r in all_run_results]
    precs = [r['test_precision_macro'] for r in all_run_results]
    recs = [r['test_recall_macro'] for r in all_run_results]
    f1s = [r['test_f1_macro'] for r in all_run_results]
    run_ids = [r['run_id'] for r in all_run_results]

    plt.figure(figsize=(12, 7))
    plt.plot(run_ids, accs, 'o-', linewidth=2.5, markersize=8, label='Test Accuracy (%)', color='#4ECDC4')
    plt.axhline(np.mean(accs), color='#4ECDC4', linestyle='--', alpha=0.6,
                label=f'Mean = {np.mean(accs):.2f}% ± {np.std(accs):.2f}')
    plt.xlabel('Run #'); plt.ylabel('Test Accuracy (%)')
    plt.title('Test Accuracy Across 10 Independent Holdout Runs (80/20)', fontsize=15, fontweight='bold')
    plt.xticks(run_ids)
    plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{out_dir}/00_all_runs_test_accuracy.png', dpi=250, facecolor='white')
    plt.close()

    plt.figure(figsize=(12, 7))
    width = 0.25
    x = np.arange(len(run_ids))
    plt.bar(x - width, precs, width, label='Precision', color='#FF6B6B', alpha=0.85, edgecolor='black')
    plt.bar(x, recs, width, label='Recall', color='#4ECDC4', alpha=0.85, edgecolor='black')
    plt.bar(x + width, f1s, width, label='F1-Score', color='#45B7D1', alpha=0.85, edgecolor='black')
    plt.xticks(x, [f'Run {r}' for r in run_ids])
    plt.ylabel('Score (macro avg)')
    plt.title('Precision / Recall / F1 Across 10 Holdout Runs', fontsize=15, fontweight='bold')
    plt.legend(); plt.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(f'{out_dir}/00_all_runs_metrics.png', dpi=250, facecolor='white')
    plt.close()

    print("✅ تم حفظ: 00_all_runs_test_accuracy.png, 00_all_runs_metrics.png")


# ==========================================================================
# ONE full training run (Holdout 80/20)
# ==========================================================================
def run_single_training(run_id, full_dataset, out_dir, num_epochs=60, batch_size=32,
                         learning_rate=0.0005, image_size=128):
    print("\n" + "=" * 70)
    print(f"🚀 RUN {run_id}/10 STARTING (Holdout 80/20, seed={42 + run_id})")
    print("=" * 70)

    seed = 42 + run_id
    set_seed(seed)

    train_loader, test_loader, class_names = create_holdout_loaders(
        full_dataset, batch_size=batch_size, image_size=image_size,
        train_split=0.8, seed=seed
    )

    model = UltraLightPatentBrainTumorCNN(num_classes=4).to(device)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"📊 Model parameters: {total_params:,}")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    train_losses, train_accs, test_acc_per_epoch = [], [], []
    best_test_acc = 0.0
    best_state = None
    final_test_preds, final_test_targets, final_test_probs = None, None, None

    start_time = time.time()

    for epoch in range(num_epochs):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion,
                                             device, epoch, num_epochs, run_id)

        # التقييم في كل epoch يتم على مجموعة الـ Testing (20%) نفسها
        # لأن منهجية Holdout المطلوبة لا تحتوي Validation منفصل
        test_loss, test_acc, test_preds, test_targets, test_probs = evaluate(
            model, test_loader, criterion, device,
            desc=f"Run {run_id} | Epoch {epoch+1}/{num_epochs} - Test"
        )
        scheduler.step()

        train_losses.append(train_loss)
        train_accs.append(train_acc)
        test_acc_per_epoch.append(test_acc)

        print(f"📈 Train: Loss={train_loss:.4f} Acc={train_acc:.2f}%  |  "
              f"🧪 Test: Loss={test_loss:.4f} Acc={test_acc:.2f}%  |  "
              f"LR={optimizer.param_groups[0]['lr']:.6f}")

        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            final_test_preds, final_test_targets, final_test_probs = test_preds, test_targets, test_probs

        if device.type == 'cuda':
            torch.cuda.empty_cache()
        gc.collect()

    training_time = time.time() - start_time

    if best_state is not None:
        model.load_state_dict(best_state)

    test_precision = precision_score(final_test_targets, final_test_preds, average='macro', zero_division=0)
    test_recall = recall_score(final_test_targets, final_test_preds, average='macro', zero_division=0)
    test_f1 = f1_score(final_test_targets, final_test_preds, average='macro', zero_division=0)

    print(f"\n🏁 RUN {run_id} RESULTS (BEST EPOCH, TEST = 20% HOLDOUT)")
    print(f"   Accuracy : {best_test_acc:.2f}%")
    print(f"   Precision: {test_precision:.4f}")
    print(f"   Recall   : {test_recall:.4f}")
    print(f"   F1-Score : {test_f1:.4f}")
    print(classification_report(final_test_targets, final_test_preds, target_names=class_names, digits=3))

    torch.save({
        'run_id': run_id,
        'model_state_dict': best_state if best_state is not None else model.state_dict(),
        'test_accuracy': best_test_acc,
        'class_names': class_names,
        'seed': seed,
    }, f'{out_dir}/run{run_id}_best_model.pth')

    plot_run_summary(run_id, train_losses, train_accs, test_acc_per_epoch,
                      final_test_targets, final_test_preds, final_test_probs,
                      class_names, out_dir)

    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()

    return {
        'run_id': run_id,
        'seed': seed,
        'test_accuracy': best_test_acc,
        'test_precision_macro': test_precision,
        'test_recall_macro': test_recall,
        'test_f1_macro': test_f1,
        'training_time_sec': training_time,
        'total_params': total_params,
    }


# ==========================================================================
# MAIN: 10 RUNS مع Holdout 80/20
# ==========================================================================
def main():
    print("=" * 70)
    print("ADVANCED BRAIN TUMOR MRI TRAINING - 10 RUNS (HOLDOUT 80/20)")
    print("=" * 70)

    # ======= عدّل هذا المسار حسب مكان مجلدك =======
    DATA_DIR = "Brain Tumor MRI Dataset Big"
    # يمكن أن يحتوي المجلد على:
    #   - Training/{glioma,meningioma,notumor,pituitary}  و/أو
    #   - Testing/{glioma,meningioma,notumor,pituitary}
    # سيتم دمج كل الصور الموجودة (مثلاً الـ 6420 صورة) في مجموعة واحدة
    # ثم تطبيق Holdout 80/20 عليها بأكملها.
    # ==============================================

    OUT_DIR = "results_10runs_holdout"
    os.makedirs(OUT_DIR, exist_ok=True)

    NUM_RUNS = 10
    NUM_EPOCHS = 60
    BATCH_SIZE = 32
    LEARNING_RATE = 0.0005
    IMAGE_SIZE = 128

    if not os.path.exists(DATA_DIR):
        print(f"❌ Data directory not found: {DATA_DIR}")
        return None

    # تحميل كل الصور مرة واحدة (بدون transform - سيُطبَّق لاحقاً لكل تقسيم)
    full_dataset = BrainTumorMRIDataset(DATA_DIR, transform=None)

    if len(full_dataset) == 0:
        print("❌ لم يتم العثور على أي صور. تحقق من مسار المجلد وأسماء الفئات.")
        return None

    print(f"📂 Data directory : {DATA_DIR}")
    print(f"🔢 Total images   : {len(full_dataset)}")
    print(f"🔁 Number of runs : {NUM_RUNS}")
    print(f"🔥 Epochs per run : {NUM_EPOCHS}")
    print(f"📦 Batch size     : {BATCH_SIZE}")
    print(f"✂️ Split          : 80% Training / 20% Testing (Holdout, no separate validation)")

    all_run_results = []
    grand_start = time.time()

    for run_id in range(1, NUM_RUNS + 1):
        result = run_single_training(
            run_id=run_id,
            full_dataset=full_dataset,
            out_dir=OUT_DIR,
            num_epochs=NUM_EPOCHS,
            batch_size=BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            image_size=IMAGE_SIZE,
        )
        all_run_results.append(result)

        with open(f'{OUT_DIR}/all_runs_results.json', 'w', encoding='utf-8') as f:
            json.dump(all_run_results, f, indent=2, ensure_ascii=False)

    grand_total_time = time.time() - grand_start

    accs = [r['test_accuracy'] for r in all_run_results]
    precs = [r['test_precision_macro'] for r in all_run_results]
    recs = [r['test_recall_macro'] for r in all_run_results]
    f1s = [r['test_f1_macro'] for r in all_run_results]
    times = [r['training_time_sec'] for r in all_run_results]

    summary = {
        'methodology': 'Holdout validation, 80% train / 20% test, no separate validation set',
        'num_runs': NUM_RUNS,
        'total_images': len(full_dataset),
        'test_accuracy_mean': float(np.mean(accs)),
        'test_accuracy_std': float(np.std(accs)),
        'test_accuracy_min': float(np.min(accs)),
        'test_accuracy_max': float(np.max(accs)),
        'test_precision_macro_mean': float(np.mean(precs)),
        'test_precision_macro_std': float(np.std(precs)),
        'test_recall_macro_mean': float(np.mean(recs)),
        'test_recall_macro_std': float(np.std(recs)),
        'test_f1_macro_mean': float(np.mean(f1s)),
        'test_f1_macro_std': float(np.std(f1s)),
        'avg_training_time_sec_per_run': float(np.mean(times)),
        'grand_total_time_sec': grand_total_time,
        'per_run_results': all_run_results,
    }

    with open(f'{OUT_DIR}/final_summary_10runs.json', 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    plot_multi_run_summary(all_run_results, OUT_DIR)

    print("\n" + "=" * 70)
    print("🏆 FINAL SUMMARY — 10 HOLDOUT RUNS (80/20) COMPLETED")
    print("=" * 70)
    print(f"Total images  : {len(full_dataset)}")
    print(f"Test Accuracy : {summary['test_accuracy_mean']:.2f}% ± {summary['test_accuracy_std']:.2f}%  "
          f"(min={summary['test_accuracy_min']:.2f}%, max={summary['test_accuracy_max']:.2f}%)")
    print(f"Precision     : {summary['test_precision_macro_mean']:.4f} ± {summary['test_precision_macro_std']:.4f}")
    print(f"Recall        : {summary['test_recall_macro_mean']:.4f} ± {summary['test_recall_macro_std']:.4f}")
    print(f"F1-Score      : {summary['test_f1_macro_mean']:.4f} ± {summary['test_f1_macro_std']:.4f}")
    print(f"Total time    : {grand_total_time/60:.1f} minutes for all {NUM_RUNS} runs")
    print(f"\n💾 كل النتائج والصور محفوظة داخل: {OUT_DIR}/")
    print("   - runX_best_model.pth               (أفضل نموذج لكل Run)")
    print("   - runX_training_curves.png          (منحنيات Loss/Acc لكل Run)")
    print("   - runX_test_confusion_matrix.png    (Confusion Matrix على الـ20%)")
    print("   - runX_test_roc_curves.png          (ROC curves على الـ20%)")
    print("   - all_runs_results.json             (نتائج تراكمية)")
    print("   - final_summary_10runs.json         (المتوسط والانحراف المعياري)")
    print("   - 00_all_runs_test_accuracy.png")
    print("   - 00_all_runs_metrics.png")

    return summary


if __name__ == "__main__":
    print("🧠 ADVANCED BRAIN TUMOR MRI CLASSIFICATION — 10 RUNS (HOLDOUT 80/20)")
    print("=" * 70)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

    try:
        summary = main()
        if summary is not None:
            print("\n🎉 SUCCESS! كل الـ 10 Runs اكتملت بنجاح.")
        else:
            print("\n❌ فشل التنفيذ. تحقق من مسار البيانات.")
    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()
        print("\nTroubleshooting tips:")
        print("1. تأكد من وجود مجلد 'Brain Tumor MRI Dataset Big'")
        print("2. يجب أن يحتوي على مجلد Training و/أو Testing، كل منهما فيه:")
        print("   glioma, meningioma, notumor, pituitary")
        print("3. قلل batch_size إذا واجهت مشاكل في الذاكرة")

GPU: NVIDIA GeForce RTX 5080 Laptop GPU
Using device: cuda
🧠 ADVANCED BRAIN TUMOR MRI CLASSIFICATION — 10 RUNS (HOLDOUT 80/20)
ADVANCED BRAIN TUMOR MRI TRAINING - 10 RUNS (HOLDOUT 80/20)
📂 Scanning: Brain Tumor MRI Dataset Big\Training
📂 Scanning: Brain Tumor MRI Dataset Big\Testing
✅ تم تحميل 12064 صورة فريدة من كل المصادر المتاحة
📁 إجمالي الصور المجمّعة (كل المصادر): 12064
📊 Class Distribution (Full Dataset):
  glioma: 3773 images
  meningioma: 2729 images
  notumor: 2432 images
  pituitary: 3130 images
📂 Data directory : Brain Tumor MRI Dataset Big
🔢 Total images   : 12064
🔁 Number of runs : 10
🔥 Epochs per run : 60
📦 Batch size     : 32
✂️ Split          : 80% Training / 20% Testing (Holdout, no separate validation)

🚀 RUN 1/10 STARTING (Holdout 80/20, seed=43)
📈 Training samples (80%): 9651
🧪 Testing samples  (20%): 2413
📊 Model parameters: 659,228


Run 1 | Epoch 1/60 - Train: 100%|██████████| 301/301 [01:34<00:00,  3.19it/s, Loss=0.933, Acc=51.9%]
Run 1 | Epoch 1/60 - Test: 100%|█████████████| 76/76 [00:16<00:00,  4.60it/s, Loss=1.251, Acc=49.0%]


📈 Train: Loss=1.1292 Acc=51.87%  |  🧪 Test: Loss=1.1369 Acc=48.98%  |  LR=0.000500


Run 1 | Epoch 2/60 - Train: 100%|██████████| 301/301 [00:36<00:00,  8.20it/s, Loss=0.590, Acc=62.1%]
Run 1 | Epoch 2/60 - Test: 100%|█████████████| 76/76 [00:04<00:00, 16.51it/s, Loss=0.735, Acc=71.3%]


📈 Train: Loss=0.9103 Acc=62.15%  |  🧪 Test: Loss=0.7192 Acc=71.28%  |  LR=0.000499


...

📈 Train: Loss=0.0464 Acc=98.49%  |  🧪 Test: Loss=0.0436 Acc=98.88%  |  LR=0.000001


Run 10 | Epoch 59/60 - Train: 100%|████████| 301/301 [00:36<00:00,  8.14it/s, Loss=0.085, Acc=98.5%]
Run 10 | Epoch 59/60 - Test: 100%|███████████| 76/76 [00:04<00:00, 15.83it/s, Loss=0.005, Acc=98.8%]


📈 Train: Loss=0.0479 Acc=98.53%  |  🧪 Test: Loss=0.0427 Acc=98.80%  |  LR=0.000000


Run 10 | Epoch 60/60 - Train: 100%|████████| 301/301 [00:36<00:00,  8.18it/s, Loss=0.046, Acc=98.7%]
Run 10 | Epoch 60/60 - Test: 100%|███████████| 76/76 [00:04<00:00, 16.00it/s, Loss=0.004, Acc=98.8%]


📈 Train: Loss=0.0417 Acc=98.68%  |  🧪 Test: Loss=0.0434 Acc=98.84%  |  LR=0.000000

🏁 RUN 10 RESULTS (BEST EPOCH, TEST = 20% HOLDOUT)
   Accuracy : 98.92%
   Precision: 0.9890
   Recall   : 0.9893
   F1-Score : 0.9891
              precision    recall  f1-score   support

      glioma      0.989     0.984     0.987       756
  meningioma      0.992     0.977     0.984       514
     notumor      0.981     0.998     0.989       515
   pituitary      0.994     0.998     0.996       628

    accuracy                          0.989      2413
   macro avg      0.989     0.989     0.989      2413
weighted avg      0.989     0.989     0.989      2413

✅ تم حفظ: 00_all_runs_test_accuracy.png, 00_all_runs_metrics.png

🏆 FINAL SUMMARY — 10 HOLDOUT RUNS (80/20) COMPLETED
Total images  : 12064
Test Accuracy : 98.64% ± 0.22%  (min=98.38%, max=99.13%)
Precision     : 0.9858 ± 0.0024
Recall        : 0.9867 ± 0.0021
F1-Score      : 0.9862 ± 0.0022
Total time    : 485.5 minutes for all 10 runs

💾 كل ال